# Análise Exploratória dos Dados

## Objetivo

Esta é a segunda etapa do trabalho. Aqui ainda não vamos ajustar ou comparar modelos de previsão.

O objetivo é compreender o comportamento das séries semanais de internações associadas ao vírus sincicial respiratório (VSR), construídas na etapa de tratamento dos dados.

O foco continua sendo crianças menores de dois anos, com análise em três níveis geográficos: Unidade da Federação, região e Brasil.

As análises históricas serão conduzidas principalmente com semanas maduras, isto é, semanas com tempo suficiente para receber notificações e confirmações laboratoriais. Semanas recentes serão preservadas, mas interpretadas com cautela devido ao atraso de disponibilidade dos dados.

## Etapas previstas

1. Carregar e validar o painel semanal produzido na Etapa 1.
2. Descrever o período disponível, a cobertura e a quantidade de casos por UF, região e Brasil.
3. Construir gráficos temporais para visualizar crescimento, queda e picos de casos.
4. Investigar a sazonalidade do VSR e verificar se os picos ocorrem nos mesmos períodos a cada ano.
5. Comparar o comportamento das regiões e das Unidades da Federação.
6. Avaliar semanas com zero caso, grande variabilidade e possíveis valores extremos.
7. Examinar dependência temporal, correlação entre semanas próximas e diferenças entre períodos.
8. Documentar os achados que orientarão a escolha, a configuração e a avaliação dos modelos de previsão.

## Resultado esperado

Ao final, teremos uma compreensão documentada da dinâmica temporal do VSR no Brasil.

Essa análise servirá para justificar quais modelos serão testados, quais níveis geográficos exigem maior atenção, quais períodos podem ser usados para treinamento e avaliação e quais limitações precisam ser consideradas na interpretação das previsões.

## Etapa 12 — Carregamento e validação do painel semanal

Nesta etapa, vamos carregar o painel semanal construído no tratamento dos dados e confirmar que ele mantém a estrutura esperada.

Ainda não serão produzidos gráficos ou conclusões sobre sazonalidade. Primeiro, verificaremos se existem duplicidades, quais níveis geográficos estão disponíveis e quantas semanas maduras podem ser usadas na análise histórica.

In [ ]:
# ETAPA 12 — Carregamento e validação do painel semanal para a EDA

PASTA_ETAPA_12 = OUTPUT_DIR / "etapa_12_validacao_eda"
PASTA_ETAPA_12.mkdir(parents=True, exist_ok=True)

ARQUIVO_PAINEL_EDA = (
    PROCESSED_DIR / "painel_vsr_semanal_com_maturidade.parquet"
)

if not ARQUIVO_PAINEL_EDA.exists():
    raise FileNotFoundError(
        f"Painel não encontrado: {ARQUIVO_PAINEL_EDA}"
    )

painel_eda = pd.read_parquet(ARQUIVO_PAINEL_EDA)

painel_eda["semana"] = pd.to_datetime(
    painel_eda["semana"],
    errors="coerce",
)

CHAVE_PAINEL = [
    "nivel",
    "unidade",
    "semana",
]

duplicidades_painel = int(
    painel_eda.duplicated(CHAVE_PAINEL).sum()
)

painel_eda["casos_vsr_maduros"] = painel_eda[
    "casos_vsr"
].where(
    painel_eda["semana_madura_13s"]
)

perfil_unidades_eda = (
    painel_eda
    .groupby(
        ["nivel", "unidade", "regiao"],
        as_index=False,
    )
    .agg(
        primeira_semana=("semana", "min"),
        ultima_semana=("semana", "max"),
        semanas_no_painel=("semana", "nunique"),
        semanas_com_valor=(
            "casos_vsr",
            lambda x: int(x.notna().sum()),
        ),
        semanas_maduras=(
            "semana_madura_13s",
            "sum",
        ),
        semanas_nao_maduras=(
            "semana_madura_13s",
            lambda x: int((~x).sum()),
        ),
        semanas_zero_maduras=(
            "casos_vsr_maduros",
            lambda x: int((x == 0).sum()),
        ),
        casos_vsr_maduros=(
            "casos_vsr_maduros",
            "sum",
        ),
        pico_semanal_maduro=(
            "casos_vsr_maduros",
            "max",
        ),
    )
    .sort_values(["nivel", "unidade"])
    .reset_index(drop=True)
)

controle_entrada_eda = pd.DataFrame(
    [
        {
            "linhas_painel": len(painel_eda),
            "duplicidades_na_chave": duplicidades_painel,
            "niveis_encontrados": " | ".join(
                sorted(painel_eda["nivel"].unique())
            ),
            "ufs": int(
                painel_eda.loc[
                    painel_eda["nivel"] == "UF",
                    "unidade",
                ].nunique()
            ),
            "regioes": int(
                painel_eda.loc[
                    painel_eda["nivel"] == "REGIAO",
                    "unidade",
                ].nunique()
            ),
            "semanas_total": int(
                painel_eda["semana"].nunique()
            ),
            "semanas_maduras_brasil": int(
                painel_eda.loc[
                    painel_eda["nivel"] == "BRASIL",
                    "semana_madura_13s",
                ].sum()
            ),
        }
    ]
)

perfil_unidades_eda.to_csv(
    PASTA_ETAPA_12 / "perfil_unidades_eda.csv",
    index=False,
    encoding="utf-8-sig",
)

controle_entrada_eda.to_csv(
    PASTA_ETAPA_12 / "controle_entrada_eda.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Etapa 12 concluída.")
print(f"Painel carregado: {ARQUIVO_PAINEL_EDA}")
print(f"Resultados: {PASTA_ETAPA_12}")
print(f"Duplicidades na chave: {duplicidades_painel}")
print(
    "Semanas maduras do Brasil: "
    f"{int(controle_entrada_eda['semanas_maduras_brasil'].iloc[0])}"
)

## Etapa 13 — Séries temporais do Brasil e das regiões

Nesta etapa, serão construídos os primeiros gráficos da análise exploratória.

O objetivo é observar a evolução semanal do VSR, identificar picos, comparar regiões e visualizar a fronteira entre o histórico maduro e as semanas recentes que ainda podem sofrer atualização.

In [ ]:
# ETAPA 13 — Gráficos temporais do Brasil e das regiões

PASTA_ETAPA_13 = OUTPUT_DIR / "etapa_13_series_brasil_regioes"
PASTA_ETAPA_13.mkdir(parents=True, exist_ok=True)

painel_grafico = pd.read_parquet(ARQUIVO_PAINEL_EDA)

painel_grafico["semana"] = pd.to_datetime(
    painel_grafico["semana"],
    errors="coerce",
)

painel_grafico = painel_grafico.sort_values("semana")

brasil = painel_grafico[
    painel_grafico["nivel"] == "BRASIL"
].copy()

regioes = painel_grafico[
    painel_grafico["nivel"] == "REGIAO"
].copy()

brasil_maduro = brasil[
    brasil["semana_madura_13s"]
].copy()

inicio_periodo_recente = brasil.loc[
    ~brasil["semana_madura_13s"],
    "semana",
].min()

# Resumo numérico para análise posterior
resumo_brasil_regioes = (
    painel_grafico[
        painel_grafico["nivel"].isin(["BRASIL", "REGIAO"])
        & painel_grafico["semana_madura_13s"]
    ]
    .groupby(
        ["nivel", "unidade", "regiao"],
        as_index=False,
    )
    .agg(
        casos_maduros=("casos_vsr", "sum"),
        media_semanal=("casos_vsr", "mean"),
        mediana_semanal=("casos_vsr", "median"),
        desvio_padrao=("casos_vsr", "std"),
        pico_semanal=("casos_vsr", "max"),
        semanas_zero=(
            "casos_vsr",
            lambda x: int((x == 0).sum()),
        ),
    )
)

total_brasil = resumo_brasil_regioes.loc[
    resumo_brasil_regioes["nivel"] == "BRASIL",
    "casos_maduros",
].iloc[0]

resumo_brasil_regioes["participacao_no_brasil_pct"] = (
    100
    * resumo_brasil_regioes["casos_maduros"]
    / total_brasil
).round(2)

resumo_brasil_regioes.to_csv(
    PASTA_ETAPA_13 / "resumo_brasil_regioes_eda.csv",
    index=False,
    encoding="utf-8-sig",
)

# Gráfico 1: Brasil
fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(
    brasil["semana"],
    brasil["casos_vsr"],
    color="#AAB7C4",
    linewidth=1,
    label="Casos semanais observados",
)

ax.plot(
    brasil_maduro["semana"],
    brasil_maduro["casos_vsr"].rolling(
        window=4,
        center=True,
        min_periods=1,
    ).mean(),
    color="#0B4F71",
    linewidth=2.5,
    label="Média móvel de 4 semanas (histórico maduro)",
)

if pd.notna(inicio_periodo_recente):
    ax.axvspan(
        inicio_periodo_recente,
        brasil["semana"].max(),
        color="#F4A261",
        alpha=0.22,
        label="Semanas recentes ou incompletas",
    )

ax.set_title(
    "Brasil — internações semanais por VSR em menores de dois anos",
    loc="left",
    fontweight="bold",
)
ax.set_xlabel("Semana de início dos sintomas")
ax.set_ylabel("Casos confirmados")
ax.legend()
plt.tight_layout()

fig.savefig(
    PASTA_ETAPA_13 / "serie_temporal_brasil.png",
    dpi=160,
    bbox_inches="tight",
)
plt.close(fig)

# Gráfico 2: regiões
fig, eixos = plt.subplots(
    nrows=5,
    ncols=1,
    figsize=(16, 15),
    sharex=True,
)

for eixo, regiao in zip(eixos, ORDENACAO_REGIOES):
    dados_regiao = regioes[
        regioes["unidade"] == regiao
    ].copy()

    dados_maduros = dados_regiao[
        dados_regiao["semana_madura_13s"]
    ].copy()

    eixo.plot(
        dados_regiao["semana"],
        dados_regiao["casos_vsr"],
        color="#B8C4CE",
        linewidth=0.9,
    )

    eixo.plot(
        dados_maduros["semana"],
        dados_maduros["casos_vsr"].rolling(
            window=4,
            center=True,
            min_periods=1,
        ).mean(),
        color="#1F6F8B",
        linewidth=2,
    )

    if pd.notna(inicio_periodo_recente):
        eixo.axvspan(
            inicio_periodo_recente,
            dados_regiao["semana"].max(),
            color="#F4A261",
            alpha=0.18,
        )

    eixo.set_title(regiao, loc="left", fontweight="bold")
    eixo.set_ylabel("Casos")

eixos[-1].set_xlabel("Semana de início dos sintomas")

fig.suptitle(
    "Regiões — internações semanais por VSR em menores de dois anos",
    x=0.08,
    y=1.01,
    ha="left",
    fontweight="bold",
)

plt.tight_layout()

fig.savefig(
    PASTA_ETAPA_13 / "series_temporais_regioes.png",
    dpi=160,
    bbox_inches="tight",
)
plt.close(fig)

print("Etapa 13 concluída.")
print(f"Resultados: {PASTA_ETAPA_13}")
print(
    "Início das semanas recentes: "
    f"{inicio_periodo_recente.date()}"
)

## Etapa 14 — Sazonalidade e picos anuais

Nesta etapa, vamos comparar semanas equivalentes entre temporadas completas.

O objetivo é verificar se existe um período do ano em que as internações por VSR tendem a aumentar, medir a variação entre temporadas e identificar em quais semanas ocorrem os picos anuais no Brasil e nas regiões.

In [ ]:
# ETAPA 14 — Sazonalidade por semana epidemiológica

PASTA_ETAPA_14 = OUTPUT_DIR / "etapa_14_sazonalidade"
PASTA_ETAPA_14.mkdir(parents=True, exist_ok=True)

painel_sazonal = pd.read_parquet(ARQUIVO_PAINEL_EDA)

painel_sazonal["semana"] = pd.to_datetime(
    painel_sazonal["semana"],
    errors="coerce",
)

dados_sazonais = painel_sazonal[
    painel_sazonal["nivel"].isin(["BRASIL", "REGIAO"])
    & painel_sazonal["semana_madura_13s"]
].copy()

# Identifica temporadas completas: 52 ou mais semanas maduras no ano epidemiológico.
completude_temporadas = (
    dados_sazonais
    .groupby(
        ["nivel", "unidade", "ano_epi"],
        as_index=False,
    )
    .agg(
        semanas_maduras=("semana_epi", "nunique"),
        primeira_semana=("semana", "min"),
        ultima_semana=("semana", "max"),
    )
)

completude_temporadas["temporada_completa"] = (
    completude_temporadas["semanas_maduras"] >= 52
)

dados_sazonais = dados_sazonais.merge(
    completude_temporadas[
        [
            "nivel",
            "unidade",
            "ano_epi",
            "temporada_completa",
        ]
    ],
    on=["nivel", "unidade", "ano_epi"],
    how="left",
)

# Perfil sazonal somente com temporadas completas
dados_completos = dados_sazonais[
    dados_sazonais["temporada_completa"]
].copy()

perfil_sazonal = (
    dados_completos
    .groupby(
        ["nivel", "unidade", "regiao", "semana_epi"],
        as_index=False,
    )
    .agg(
        anos_observados=("ano_epi", "nunique"),
        media_casos=("casos_vsr", "mean"),
        mediana_casos=("casos_vsr", "median"),
        desvio_padrao=("casos_vsr", "std"),
        percentil_25=(
            "casos_vsr",
            lambda x: x.quantile(0.25),
        ),
        percentil_75=(
            "casos_vsr",
            lambda x: x.quantile(0.75),
        ),
    )
)

# Pico de cada temporada completa
indice_picos = (
    dados_completos
    .groupby(
        ["nivel", "unidade", "ano_epi"],
    )["casos_vsr"]
    .idxmax()
)

picos_anuais = (
    dados_completos
    .loc[indice_picos]
    [
        [
            "nivel",
            "unidade",
            "regiao",
            "ano_epi",
            "semana_epi",
            "semana",
            "casos_vsr",
        ]
    ]
    .rename(
        columns={
            "semana_epi": "semana_epi_pico",
            "semana": "data_pico",
            "casos_vsr": "casos_no_pico",
        }
    )
    .sort_values(["nivel", "unidade", "ano_epi"])
    .reset_index(drop=True)
)

perfil_sazonal.to_csv(
    PASTA_ETAPA_14 / "perfil_sazonal_brasil_regioes.csv",
    index=False,
    encoding="utf-8-sig",
)

picos_anuais.to_csv(
    PASTA_ETAPA_14 / "picos_anuais_brasil_regioes.csv",
    index=False,
    encoding="utf-8-sig",
)

completude_temporadas.to_csv(
    PASTA_ETAPA_14 / "completude_temporadas_eda.csv",
    index=False,
    encoding="utf-8-sig",
)

# Gráfico sazonal: temporadas completas em cinza e média histórica em azul.
unidades_grafico = [
    ("BRASIL", "BRASIL"),
    ("REGIAO", "NORTE"),
    ("REGIAO", "NORDESTE"),
    ("REGIAO", "CENTRO_OESTE"),
    ("REGIAO", "SUDESTE"),
    ("REGIAO", "SUL"),
]

fig, eixos = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(16, 13),
    sharex=True,
)

for eixo, (nivel, unidade) in zip(
    eixos.flatten(),
    unidades_grafico,
):
    dados_unidade = dados_completos[
        (dados_completos["nivel"] == nivel)
        & (dados_completos["unidade"] == unidade)
    ]

    for ano in sorted(dados_unidade["ano_epi"].unique()):
        serie_ano = dados_unidade[
            dados_unidade["ano_epi"] == ano
        ].sort_values("semana_epi")

        eixo.plot(
            serie_ano["semana_epi"],
            serie_ano["casos_vsr"],
            color="#B7C5D3",
            linewidth=1,
            alpha=0.8,
        )

    media_unidade = perfil_sazonal[
        (perfil_sazonal["nivel"] == nivel)
        & (perfil_sazonal["unidade"] == unidade)
    ].sort_values("semana_epi")

    eixo.plot(
        media_unidade["semana_epi"],
        media_unidade["media_casos"],
        color="#0B4F71",
        linewidth=2.5,
        label="Média entre temporadas completas",
    )

    eixo.fill_between(
        media_unidade["semana_epi"],
        media_unidade["percentil_25"],
        media_unidade["percentil_75"],
        color="#5FA8D3",
        alpha=0.25,
        label="Faixa entre P25 e P75",
    )

    eixo.set_title(unidade, loc="left", fontweight="bold")
    eixo.set_ylabel("Casos")
    eixo.set_xlim(1, 53)

eixos[0, 0].legend(loc="upper right", fontsize=8)

for eixo in eixos[-1, :]:
    eixo.set_xlabel("Semana epidemiológica")

fig.suptitle(
    "Perfil sazonal do VSR — Brasil e regiões",
    x=0.07,
    y=1.01,
    ha="left",
    fontweight="bold",
)

plt.tight_layout()

fig.savefig(
    PASTA_ETAPA_14 / "sazonalidade_brasil_regioes.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

print("Etapa 14 concluída.")
print(f"Resultados: {PASTA_ETAPA_14}")
print(
    "Temporadas completas encontradas: "
    f"{int(completude_temporadas['temporada_completa'].sum())}"
)

## Etapa 15 — Intermitência e dispersão das Unidades da Federação

Nesta etapa, vamos medir a frequência de semanas sem casos, a variabilidade e a concentração de casos em cada Unidade da Federação.

O objetivo é identificar séries mais densas e estáveis, bem como séries esparsas, com muitos zeros ou alta variabilidade.

In [ ]:
# ETAPA 15 — Intermitência e dispersão das UFs

PASTA_ETAPA_15 = OUTPUT_DIR / "etapa_15_ufs_intermitencia"
PASTA_ETAPA_15.mkdir(parents=True, exist_ok=True)

painel_ufs = pd.read_parquet(ARQUIVO_PAINEL_EDA)

painel_ufs = painel_ufs[
    (painel_ufs["nivel"] == "UF")
    & painel_ufs["semana_madura_13s"]
].copy()

perfil_ufs = (
    painel_ufs
    .groupby(
        ["unidade", "regiao"],
        as_index=False,
    )
    .agg(
        semanas=("semana", "nunique"),
        casos_totais=("casos_vsr", "sum"),
        media_semanal=("casos_vsr", "mean"),
        mediana_semanal=("casos_vsr", "median"),
        variancia_semanal=("casos_vsr", "var"),
        desvio_padrao=("casos_vsr", "std"),
        percentil_95=(
            "casos_vsr",
            lambda x: x.quantile(0.95),
        ),
        pico_semanal=("casos_vsr", "max"),
        semanas_zero=(
            "casos_vsr",
            lambda x: int((x == 0).sum()),
        ),
    )
)

perfil_ufs["percentual_zero"] = (
    100
    * perfil_ufs["semanas_zero"]
    / perfil_ufs["semanas"]
).round(2)

perfil_ufs["indice_dispersao"] = (
    perfil_ufs["variancia_semanal"]
    / perfil_ufs["media_semanal"]
)

perfil_ufs["coeficiente_variacao"] = (
    perfil_ufs["desvio_padrao"]
    / perfil_ufs["media_semanal"]
)

perfil_ufs = perfil_ufs.sort_values(
    ["percentual_zero", "casos_totais"],
    ascending=[False, True],
).reset_index(drop=True)

perfil_ufs.to_csv(
    PASTA_ETAPA_15 / "perfil_ufs_intermitencia.csv",
    index=False,
    encoding="utf-8-sig",
)

CORES_REGIAO = {
    "NORTE": "#4C78A8",
    "NORDESTE": "#F58518",
    "CENTRO_OESTE": "#54A24B",
    "SUDESTE": "#E45756",
    "SUL": "#B279A2",
}

# Gráfico 1: volume total e proporção de semanas sem casos
fig, ax = plt.subplots(figsize=(12, 8))

for regiao in sorted(perfil_ufs["regiao"].unique()):
    dados_regiao = perfil_ufs[
        perfil_ufs["regiao"] == regiao
    ]

    ax.scatter(
        dados_regiao["percentual_zero"],
        dados_regiao["casos_totais"],
        label=regiao,
        color=CORES_REGIAO[regiao],
        s=70,
        alpha=0.85,
    )

    for linha in dados_regiao.itertuples(index=False):
        ax.annotate(
            linha.unidade,
            (
                linha.percentual_zero,
                linha.casos_totais,
            ),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=8,
        )

ax.set_yscale("log")
ax.set_xlabel("Semanas maduras sem casos (%)")
ax.set_ylabel("Casos maduros acumulados — escala logarítmica")
ax.set_title(
    "UFs — volume de casos e intermitência",
    loc="left",
    fontweight="bold",
)
ax.legend(title="Região")
plt.tight_layout()

fig.savefig(
    PASTA_ETAPA_15 / "ufs_volume_intermitencia.png",
    dpi=160,
    bbox_inches="tight",
)
plt.close(fig)

# Gráfico 2: percentual de semanas zero por UF
perfil_ordenado = perfil_ufs.sort_values(
    "percentual_zero",
    ascending=True,
)

fig, ax = plt.subplots(figsize=(12, 9))

ax.barh(
    perfil_ordenado["unidade"],
    perfil_ordenado["percentual_zero"],
    color=[
        CORES_REGIAO[regiao]
        for regiao in perfil_ordenado["regiao"]
    ],
)

ax.set_xlabel("Semanas maduras sem casos (%)")
ax.set_ylabel("Unidade da Federação")
ax.set_title(
    "Intermitência semanal por UF",
    loc="left",
    fontweight="bold",
)

plt.tight_layout()

fig.savefig(
    PASTA_ETAPA_15 / "ufs_percentual_semanas_zero.png",
    dpi=160,
    bbox_inches="tight",
)
plt.close(fig)

print("Etapa 15 concluída.")
print(f"Resultados: {PASTA_ETAPA_15}")
print(
    "UFs com mais de 50% de semanas zero: "
    f"{int((perfil_ufs['percentual_zero'] > 50).sum())}"
)

## Etapa 16 — Comparação entre temporadas e períodos

Nesta etapa, vamos comparar o total de casos, a média semanal e os picos entre temporadas completas.

O objetivo é identificar mudanças de nível ao longo dos anos e distinguir o período pré-pandemia, o período de pandemia e os anos posteriores.

Essa análise é descritiva: diferenças observadas não serão interpretadas automaticamente como efeito de uma causa específica.

In [ ]:
# ETAPA 16 — Comparação entre temporadas e períodos

PASTA_ETAPA_16 = OUTPUT_DIR / "etapa_16_temporadas_periodos"
PASTA_ETAPA_16.mkdir(parents=True, exist_ok=True)

painel_temporadas = pd.read_parquet(ARQUIVO_PAINEL_EDA)

painel_temporadas["semana"] = pd.to_datetime(
    painel_temporadas["semana"],
    errors="coerce",
)

dados_temporadas = painel_temporadas[
    painel_temporadas["nivel"].isin(["BRASIL", "REGIAO"])
    & painel_temporadas["semana_madura_13s"]
].copy()

# Mantém apenas anos epidemiológicos completos.
completude = (
    dados_temporadas
    .groupby(
        ["nivel", "unidade", "ano_epi"],
        as_index=False,
    )
    .agg(
        semanas_maduras=("semana_epi", "nunique"),
    )
)

completude["temporada_completa"] = (
    completude["semanas_maduras"] >= 52
)

dados_temporadas = dados_temporadas.merge(
    completude,
    on=["nivel", "unidade", "ano_epi"],
    how="left",
)

dados_completos = dados_temporadas[
    dados_temporadas["temporada_completa"]
].copy()

def classificar_periodo(ano_epi):
    if ano_epi == 2019:
        return "PRE_PANDEMIA"
    if ano_epi in [2020, 2021]:
        return "PANDEMIA"
    return "POS_PANDEMIA"

dados_completos["periodo_analitico"] = (
    dados_completos["ano_epi"]
    .apply(classificar_periodo)
)

resumo_temporadas = (
    dados_completos
    .groupby(
        [
            "nivel",
            "unidade",
            "regiao",
            "ano_epi",
            "periodo_analitico",
        ],
        as_index=False,
    )
    .agg(
        casos_na_temporada=("casos_vsr", "sum"),
        media_semanal=("casos_vsr", "mean"),
        mediana_semanal=("casos_vsr", "median"),
        pico_semanal=("casos_vsr", "max"),
        semanas_zero=(
            "casos_vsr",
            lambda x: int((x == 0).sum()),
        ),
    )
)

comparacao_periodos = (
    resumo_temporadas
    .groupby(
        [
            "nivel",
            "unidade",
            "regiao",
            "periodo_analitico",
        ],
        as_index=False,
    )
    .agg(
        temporadas=("ano_epi", "nunique"),
        media_casos_temporada=(
            "casos_na_temporada",
            "mean",
        ),
        media_pico_semanal=(
            "pico_semanal",
            "mean",
        ),
        media_semanas_zero=(
            "semanas_zero",
            "mean",
        ),
    )
)

resumo_temporadas.to_csv(
    PASTA_ETAPA_16 / "resumo_temporadas_brasil_regioes.csv",
    index=False,
    encoding="utf-8-sig",
)

comparacao_periodos.to_csv(
    PASTA_ETAPA_16 / "comparacao_periodos_brasil_regioes.csv",
    index=False,
    encoding="utf-8-sig",
)

CORES_PERIODO = {
    "PRE_PANDEMIA": "#4C78A8",
    "PANDEMIA": "#F58518",
    "POS_PANDEMIA": "#54A24B",
}

unidades_grafico = [
    ("BRASIL", "BRASIL"),
    ("REGIAO", "NORTE"),
    ("REGIAO", "NORDESTE"),
    ("REGIAO", "CENTRO_OESTE"),
    ("REGIAO", "SUDESTE"),
    ("REGIAO", "SUL"),
]

fig, eixos = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(16, 13),
)

for eixo, (nivel, unidade) in zip(
    eixos.flatten(),
    unidades_grafico,
):
    dados_unidade = resumo_temporadas[
        (resumo_temporadas["nivel"] == nivel)
        & (resumo_temporadas["unidade"] == unidade)
    ].sort_values("ano_epi")

    cores = [
        CORES_PERIODO[periodo]
        for periodo in dados_unidade["periodo_analitico"]
    ]

    eixo.bar(
        dados_unidade["ano_epi"].astype(str),
        dados_unidade["casos_na_temporada"],
        color=cores,
    )

    eixo.set_title(unidade, loc="left", fontweight="bold")
    eixo.set_ylabel("Casos na temporada")
    eixo.set_xlabel("Ano epidemiológico")

fig.suptitle(
    "Total de casos por temporada completa",
    x=0.07,
    y=1.01,
    ha="left",
    fontweight="bold",
)

plt.tight_layout()

fig.savefig(
    PASTA_ETAPA_16 / "totais_temporadas_brasil_regioes.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

print("Etapa 16 concluída.")
print(f"Resultados: {PASTA_ETAPA_16}")
print(
    "Temporadas completas analisadas: "
    f"{sorted(dados_completos['ano_epi'].unique())}"
)

## Etapa 17 — Dependência temporal e autocorrelação

Nesta etapa, vamos medir a relação entre os casos observados em uma semana e os casos registrados em semanas anteriores.

O objetivo é verificar se a série apresenta persistência de curto prazo e repetição aproximada do comportamento anual, especialmente em defasagens próximas de 52 semanas.

In [ ]:
# ETAPA 17 — Dependência temporal e autocorrelação

PASTA_ETAPA_17 = OUTPUT_DIR / "etapa_17_autocorrelacao"
PASTA_ETAPA_17.mkdir(parents=True, exist_ok=True)

painel_autocorrelacao = pd.read_parquet(ARQUIVO_PAINEL_EDA)

painel_autocorrelacao["semana"] = pd.to_datetime(
    painel_autocorrelacao["semana"],
    errors="coerce",
)

dados_autocorrelacao = painel_autocorrelacao[
    painel_autocorrelacao["nivel"].isin(["BRASIL", "REGIAO"])
    & painel_autocorrelacao["semana_madura_13s"]
].copy()

CHAVE_AUTOCORRELACAO = [
    "nivel",
    "unidade",
    "semana",
]

if dados_autocorrelacao.duplicated(
    CHAVE_AUTOCORRELACAO
).any():
    raise RuntimeError(
        "Foram encontradas duplicidades nas séries de autocorrelação."
    )

if dados_autocorrelacao["casos_vsr"].isna().any():
    raise RuntimeError(
        "Existem semanas maduras sem valor de casos."
    )

unidades_esperadas = {
    "BRASIL",
    "NORTE",
    "NORDESTE",
    "CENTRO_OESTE",
    "SUDESTE",
    "SUL",
}

unidades_encontradas = set(
    dados_autocorrelacao["unidade"].unique()
)

if unidades_encontradas != unidades_esperadas:
    raise RuntimeError(
        "As unidades encontradas não correspondem a Brasil e cinco regiões."
    )

# Confirma que não existem lacunas entre semanas maduras.
for unidade, grupo in dados_autocorrelacao.groupby("unidade"):
    diferencas = (
        grupo
        .sort_values("semana")["semana"]
        .diff()
        .dropna()
        .dt.days
    )

    if not (diferencas == 7).all():
        raise RuntimeError(
            f"A série de {unidade} possui lacunas entre semanas."
        )

def calcular_autocorrelacao(serie, max_defasagem):
    valores = pd.Series(
        serie,
        dtype="float64",
    ).reset_index(drop=True)

    return pd.DataFrame(
        {
            "defasagem_semanas": range(
                1,
                max_defasagem + 1,
            ),
            "autocorrelacao": [
                valores.autocorr(lag=defasagem)
                for defasagem in range(
                    1,
                    max_defasagem + 1,
                )
            ],
        }
    )

MAX_DEFASAGEM = 104
lista_autocorrelacoes = []

for (nivel, unidade, regiao), grupo in (
    dados_autocorrelacao
    .groupby(
        ["nivel", "unidade", "regiao"],
        sort=False,
    )
):
    grupo = grupo.sort_values("semana").reset_index(drop=True)

    resultado_unidade = calcular_autocorrelacao(
        grupo["casos_vsr"],
        MAX_DEFASAGEM,
    )

    resultado_unidade["nivel"] = nivel
    resultado_unidade["unidade"] = unidade
    resultado_unidade["regiao"] = regiao
    resultado_unidade["semanas_analisadas"] = len(grupo)

    lista_autocorrelacoes.append(resultado_unidade)

autocorrelacoes = pd.concat(
    lista_autocorrelacoes,
    ignore_index=True,
)[
    [
        "nivel",
        "unidade",
        "regiao",
        "semanas_analisadas",
        "defasagem_semanas",
        "autocorrelacao",
    ]
]

DEFASAGENS_RESUMO = [
    1,
    2,
    4,
    13,
    26,
    52,
    53,
    104,
]

resumo_autocorrelacao = (
    autocorrelacoes[
        autocorrelacoes["defasagem_semanas"].isin(
            DEFASAGENS_RESUMO
        )
    ]
    .pivot(
        index=[
            "nivel",
            "unidade",
            "regiao",
            "semanas_analisadas",
        ],
        columns="defasagem_semanas",
        values="autocorrelacao",
    )
    .reset_index()
    .rename(
        columns={
            defasagem: (
                f"acf_defasagem_{defasagem}_semanas"
            )
            for defasagem in DEFASAGENS_RESUMO
        }
    )
)

autocorrelacoes.to_csv(
    PASTA_ETAPA_17 / "autocorrelacoes_brasil_regioes.csv",
    index=False,
    encoding="utf-8-sig",
)

resumo_autocorrelacao.to_csv(
    PASTA_ETAPA_17 / "resumo_autocorrelacao_brasil_regioes.csv",
    index=False,
    encoding="utf-8-sig",
)

unidades_grafico = [
    ("BRASIL", "BRASIL"),
    ("REGIAO", "NORTE"),
    ("REGIAO", "NORDESTE"),
    ("REGIAO", "CENTRO_OESTE"),
    ("REGIAO", "SUDESTE"),
    ("REGIAO", "SUL"),
]

fig, eixos = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(16, 13),
    sharex=True,
    sharey=True,
)

for eixo, (nivel, unidade) in zip(
    eixos.flatten(),
    unidades_grafico,
):
    dados_unidade = autocorrelacoes[
        (autocorrelacoes["nivel"] == nivel)
        & (autocorrelacoes["unidade"] == unidade)
    ].sort_values("defasagem_semanas")

    eixo.vlines(
        dados_unidade["defasagem_semanas"],
        0,
        dados_unidade["autocorrelacao"],
        color="#1F6F8B",
        linewidth=1.1,
    )

    eixo.scatter(
        dados_unidade["defasagem_semanas"],
        dados_unidade["autocorrelacao"],
        color="#1F6F8B",
        s=12,
    )

    eixo.axhline(
        0,
        color="#4A4A4A",
        linewidth=0.8,
    )

    eixo.axvline(
        52,
        color="#D1495B",
        linestyle="--",
        linewidth=1,
    )

    eixo.set_title(
        unidade,
        loc="left",
        fontweight="bold",
    )

    eixo.set_ylabel("Autocorrelação")
    eixo.set_ylim(-1, 1)
    eixo.grid(axis="y", alpha=0.25)

for eixo in eixos[-1, :]:
    eixo.set_xlabel("Defasagem em semanas")

fig.suptitle(
    "Autocorrelação dos casos semanais maduros de VSR",
    x=0.07,
    y=1.01,
    ha="left",
    fontweight="bold",
)

plt.tight_layout()

fig.savefig(
    PASTA_ETAPA_17 / "acf_brasil_regioes.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

print("Etapa 17 concluída.")
print(f"Resultados: {PASTA_ETAPA_17}")
print(
    "Semanas maduras analisadas: "
    f"{dados_autocorrelacao['semana'].nunique()}"
)

## Etapa 18 — Evolução do nível e da variabilidade

Nesta etapa, vamos observar se o nível médio de casos e a variabilidade semanal permanecem semelhantes ou mudam ao longo do histórico.

O objetivo é verificar se períodos com mais casos também apresentam maior oscilação. Essa informação ajudará a decidir, posteriormente, se será necessário testar transformações ou modelos que acomodem mudanças de nível e variância.

In [ ]:
# ETAPA 18 — Evolução do nível e da variabilidade

PASTA_ETAPA_18 = OUTPUT_DIR / "etapa_18_nivel_variabilidade"
PASTA_ETAPA_18.mkdir(parents=True, exist_ok=True)

painel_tendencia = pd.read_parquet(ARQUIVO_PAINEL_EDA)

painel_tendencia["semana"] = pd.to_datetime(
    painel_tendencia["semana"],
    errors="coerce",
)

dados_tendencia = painel_tendencia[
    painel_tendencia["nivel"].isin(["BRASIL", "REGIAO"])
    & painel_tendencia["semana_madura_13s"]
].copy()

if dados_tendencia["casos_vsr"].isna().any():
    raise RuntimeError(
        "Existem semanas maduras sem valor de casos."
    )

# Identifica temporadas completas para a comparação numérica.
completude_tendencia = (
    dados_tendencia
    .groupby(
        ["nivel", "unidade", "ano_epi"],
        as_index=False,
    )
    .agg(
        semanas_maduras=("semana_epi", "nunique"),
    )
)

completude_tendencia["temporada_completa"] = (
    completude_tendencia["semanas_maduras"] >= 52
)

dados_tendencia = dados_tendencia.merge(
    completude_tendencia,
    on=["nivel", "unidade", "ano_epi"],
    how="left",
)

dados_temporadas_completas = dados_tendencia[
    dados_tendencia["temporada_completa"]
].copy()

resumo_variabilidade = (
    dados_temporadas_completas
    .groupby(
        ["nivel", "unidade", "regiao", "ano_epi"],
        as_index=False,
    )
    .agg(
        semanas=("semana", "nunique"),
        casos_na_temporada=("casos_vsr", "sum"),
        media_semanal=("casos_vsr", "mean"),
        mediana_semanal=("casos_vsr", "median"),
        desvio_padrao_semanal=("casos_vsr", "std"),
        variancia_semanal=("casos_vsr", "var"),
        percentil_25=(
            "casos_vsr",
            lambda x: x.quantile(0.25),
        ),
        percentil_75=(
            "casos_vsr",
            lambda x: x.quantile(0.75),
        ),
        pico_semanal=("casos_vsr", "max"),
    )
)

resumo_variabilidade["coeficiente_variacao"] = (
    resumo_variabilidade["desvio_padrao_semanal"]
    / resumo_variabilidade["media_semanal"]
)

resumo_variabilidade = resumo_variabilidade.sort_values(
    ["nivel", "unidade", "ano_epi"]
).reset_index(drop=True)

resumo_variabilidade.to_csv(
    PASTA_ETAPA_18
    / "resumo_variabilidade_temporadas_brasil_regioes.csv",
    index=False,
    encoding="utf-8-sig",
)

# Médias e desvios móveis calculados apenas para visualização.
dados_tendencia = dados_tendencia.sort_values(
    ["nivel", "unidade", "semana"]
).reset_index(drop=True)

dados_tendencia["media_movel_13s"] = (
    dados_tendencia
    .groupby(["nivel", "unidade"])["casos_vsr"]
    .transform(
        lambda x: x.rolling(
            window=13,
            min_periods=8,
        ).mean()
    )
)

dados_tendencia["desvio_movel_13s"] = (
    dados_tendencia
    .groupby(["nivel", "unidade"])["casos_vsr"]
    .transform(
        lambda x: x.rolling(
            window=13,
            min_periods=8,
        ).std()
    )
)

unidades_grafico = [
    ("BRASIL", "BRASIL"),
    ("REGIAO", "NORTE"),
    ("REGIAO", "NORDESTE"),
    ("REGIAO", "CENTRO_OESTE"),
    ("REGIAO", "SUDESTE"),
    ("REGIAO", "SUL"),
]

# Gráfico 1: nível dos casos.
fig, eixos = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(16, 13),
    sharex=True,
)

for eixo, (nivel, unidade) in zip(
    eixos.flatten(),
    unidades_grafico,
):
    dados_unidade = dados_tendencia[
        (dados_tendencia["nivel"] == nivel)
        & (dados_tendencia["unidade"] == unidade)
    ]

    eixo.plot(
        dados_unidade["semana"],
        dados_unidade["casos_vsr"],
        color="#B7C5D3",
        linewidth=0.8,
        alpha=0.7,
    )

    eixo.plot(
        dados_unidade["semana"],
        dados_unidade["media_movel_13s"],
        color="#0B4F71",
        linewidth=2.2,
    )

    eixo.set_title(
        unidade,
        loc="left",
        fontweight="bold",
    )
    eixo.set_ylabel("Casos")

for eixo in eixos[-1, :]:
    eixo.set_xlabel("Semana de início dos sintomas")

fig.suptitle(
    "Nível semanal dos casos e média móvel de 13 semanas",
    x=0.07,
    y=1.01,
    ha="left",
    fontweight="bold",
)

plt.tight_layout()

fig.savefig(
    PASTA_ETAPA_18
    / "nivel_movel_13s_brasil_regioes.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

# Gráfico 2: variabilidade dos casos.
fig, eixos = plt.subplots(
    nrows=3,
    ncols=2,
    figsize=(16, 13),
    sharex=True,
)

for eixo, (nivel, unidade) in zip(
    eixos.flatten(),
    unidades_grafico,
):
    dados_unidade = dados_tendencia[
        (dados_tendencia["nivel"] == nivel)
        & (dados_tendencia["unidade"] == unidade)
    ]

    eixo.plot(
        dados_unidade["semana"],
        dados_unidade["desvio_movel_13s"],
        color="#D1495B",
        linewidth=2,
    )

    eixo.set_title(
        unidade,
        loc="left",
        fontweight="bold",
    )
    eixo.set_ylabel("Desvio-padrão móvel")

for eixo in eixos[-1, :]:
    eixo.set_xlabel("Semana de início dos sintomas")

fig.suptitle(
    "Variabilidade semanal móvel em janelas de 13 semanas",
    x=0.07,
    y=1.01,
    ha="left",
    fontweight="bold",
)

plt.tight_layout()

fig.savefig(
    PASTA_ETAPA_18
    / "variabilidade_movel_13s_brasil_regioes.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

print("Etapa 18 concluída.")
print(f"Resultados: {PASTA_ETAPA_18}")
print(
    "Temporadas completas analisadas: "
    f"{sorted(dados_temporadas_completas['ano_epi'].unique())}"
)

## Etapa 19 — Calendário de validação temporal

Nesta etapa, vamos definir os cortes históricos que serão usados posteriormente para testar previsões sem utilizar informações do futuro.

O objetivo é criar um calendário de validação com diferentes horizontes de previsão, preservando a ordem cronológica da série e garantindo uma quantidade suficiente de histórico para cada ajuste.

In [ ]:
# ETAPA 19 — Calendário de validação temporal

PASTA_ETAPA_19 = OUTPUT_DIR / "etapa_19_validacao_temporal"
PASTA_ETAPA_19.mkdir(parents=True, exist_ok=True)

painel_validacao = pd.read_parquet(ARQUIVO_PAINEL_EDA)

painel_validacao["semana"] = pd.to_datetime(
    painel_validacao["semana"],
    errors="coerce",
)

dados_validacao = painel_validacao[
    painel_validacao["nivel"].isin(["BRASIL", "REGIAO"])
    & painel_validacao["semana_madura_13s"]
].copy()

CHAVE_VALIDACAO = [
    "nivel",
    "unidade",
    "semana",
]

if dados_validacao.duplicated(CHAVE_VALIDACAO).any():
    raise RuntimeError(
        "Existem duplicidades na base usada para validação temporal."
    )

if dados_validacao["casos_vsr"].isna().any():
    raise RuntimeError(
        "Existem semanas maduras sem valor de casos."
    )

semanas_disponiveis = sorted(
    dados_validacao["semana"].unique()
)

for unidade, grupo in dados_validacao.groupby("unidade"):
    diferencas = (
        grupo
        .sort_values("semana")["semana"]
        .diff()
        .dropna()
        .dt.days
    )

    if not (diferencas == 7).all():
        raise RuntimeError(
            f"A série de {unidade} possui lacunas entre semanas."
        )

# Três anos de histórico permitem observar pelo menos
# duas repetições completas do ciclo anual antes da previsão.
MINIMO_SEMANAS_TREINO = 156

# Horizontes operacionais: uma semana, um mês,
# dois meses e aproximadamente um trimestre.
HORIZONTES = [1, 4, 8, 13]

# Novo corte a cada quatro semanas, mais o último corte possível.
PASSO_ENTRE_CORTES = 4
MAX_HORIZONTE = max(HORIZONTES)

ultimo_indice_origem = (
    len(semanas_disponiveis)
    - 1
    - MAX_HORIZONTE
)

indices_origem = list(
    range(
        MINIMO_SEMANAS_TREINO - 1,
        ultimo_indice_origem + 1,
        PASSO_ENTRE_CORTES,
    )
)

if indices_origem[-1] != ultimo_indice_origem:
    indices_origem.append(ultimo_indice_origem)

if len(indices_origem) == 0:
    raise RuntimeError(
        "Não há histórico suficiente para criar os cortes de validação."
    )

lista_calendario = []

for (nivel, unidade, regiao), grupo in (
    dados_validacao
    .groupby(
        ["nivel", "unidade", "regiao"],
        sort=False,
    )
):
    grupo = grupo.sort_values("semana").reset_index(drop=True)

    for indice_origem in indices_origem:
        data_origem = grupo.loc[
            indice_origem,
            "semana",
        ]

        semanas_treino = indice_origem + 1

        for horizonte in HORIZONTES:
            indice_alvo = indice_origem + horizonte

            lista_calendario.append(
                {
                    "nivel": nivel,
                    "unidade": unidade,
                    "regiao": regiao,
                    "data_inicio_treino": grupo.loc[
                        0,
                        "semana",
                    ],
                    "data_origem": data_origem,
                    "semanas_treino": semanas_treino,
                    "horizonte_semanas": horizonte,
                    "data_alvo": grupo.loc[
                        indice_alvo,
                        "semana",
                    ],
                    "casos_observados_alvo": grupo.loc[
                        indice_alvo,
                        "casos_vsr",
                    ],
                }
            )

calendario_validacao = pd.DataFrame(
    lista_calendario
).sort_values(
    [
        "nivel",
        "unidade",
        "data_origem",
        "horizonte_semanas",
    ]
).reset_index(drop=True)

resumo_validacao = (
    calendario_validacao
    .groupby(
        [
            "nivel",
            "unidade",
            "regiao",
            "horizonte_semanas",
        ],
        as_index=False,
    )
    .agg(
        cortes_de_validacao=("data_origem", "nunique"),
        previsoes_avaliadas=("data_alvo", "size"),
        primeira_origem=("data_origem", "min"),
        ultima_origem=("data_origem", "max"),
        primeiro_alvo=("data_alvo", "min"),
        ultimo_alvo=("data_alvo", "max"),
        minimo_semanas_treino=("semanas_treino", "min"),
        maximo_semanas_treino=("semanas_treino", "max"),
    )
)

calendario_validacao.to_csv(
    PASTA_ETAPA_19 / "calendario_validacao_temporal.csv",
    index=False,
    encoding="utf-8-sig",
)

resumo_validacao.to_csv(
    PASTA_ETAPA_19 / "resumo_validacao_temporal.csv",
    index=False,
    encoding="utf-8-sig",
)

# Visualização dos cortes para a série nacional.
brasil = dados_validacao[
    dados_validacao["nivel"] == "BRASIL"
].sort_values("semana")

origens_brasil = (
    calendario_validacao[
        calendario_validacao["unidade"] == "BRASIL"
    ]["data_origem"]
    .drop_duplicates()
    .sort_values()
)

fig, ax = plt.subplots(figsize=(16, 6))

ax.plot(
    brasil["semana"],
    brasil["casos_vsr"],
    color="#B7C5D3",
    linewidth=1,
    label="Casos semanais maduros",
)

ax.scatter(
    origens_brasil,
    brasil.set_index("semana").loc[
        origens_brasil,
        "casos_vsr",
    ],
    color="#0B4F71",
    s=20,
    label="Cortes de validação",
    zorder=3,
)

ax.axvline(
    origens_brasil.min(),
    color="#D1495B",
    linestyle="--",
    linewidth=1.2,
    label="Primeiro corte",
)

ax.set_title(
    "Brasil — cortes temporais para validação de previsões",
    loc="left",
    fontweight="bold",
)

ax.set_xlabel("Semana de início dos sintomas")
ax.set_ylabel("Casos confirmados")
ax.legend()

plt.tight_layout()

fig.savefig(
    PASTA_ETAPA_19
    / "calendario_validacao_brasil.png",
    dpi=160,
    bbox_inches="tight",
)

plt.close(fig)

print("Etapa 19 concluída.")
print(f"Resultados: {PASTA_ETAPA_19}")
print(
    "Cortes de validação por série: "
    f"{len(indices_origem)}"
)
print(
    "Horizontes de previsão: "
    f"{HORIZONTES}"
)